# DDS-Edge: cross-validation theo tài xế (10 lớp, Colab)

**Vì sao cần cross-validation:** ở các lần chạy trước, mỗi mô hình chỉ được chấm trên **4 tài xế test**. Điểm dao động nhiều theo việc tài xế nào rơi vào tập test (độ lệch chuẩn khoảng 0,03), nên các mô hình cách nhau dưới 0,04 **không phân biệt được**. Cross-validation theo tài xế chia 26 tài xế thành **5 nhóm**; mỗi lần train, một nhóm (5 đến 6 tài xế) làm test, 4 tài xế làm val, còn lại (16 đến 17 tài xế) làm train. **Mỗi tài xế làm test đúng một lần**, và test mỗi fold có 3.600 đến 4.500 ảnh.

**Quan trọng:** cả ba mô hình dùng **cùng 5 fold** (cùng seed nên cùng cách chia). Nhờ vậy so sánh được **theo cặp**: trên mỗi fold, mô hình A hơn hay kém mô hình B bao nhiêu. Cách so sánh này loại bỏ phần lớn nhiễu do "tài xế dễ hay khó" và chính xác hơn so sánh hai con số trung bình.

**Đã có sẵn:** EfficientNet-Lite0 (`effl0_f10_cv0` đến `cv4`): macro-F1 trung bình **0,863 ± 0,032**. Notebook này chạy **MobileNetV3-Small** và **ResNet18** (tuỳ chọn ResNet50).

| Phần | Cell | Việc | Thời gian ước tính* |
|---|---|---|---|
| A | 1 đến 5 | Chuẩn bị môi trường | 5 phút |
| B | 6 | Xem 5 fold chia tài xế thế nào | 1 phút |
| C | 7 đến 11 | **MobileNetV3-Small, 5 fold** | 5 × 15 = khoảng 75 phút |
| D | 12 đến 16 | **ResNet18, 5 fold** | 5 × 8 = khoảng 40 phút |
| E | 17 | (Tuỳ chọn) ResNet50, 5 fold | rất lâu |
| F | 18 đến 19 | Tổng hợp và so sánh theo cặp | 1 phút |

\*Từ thời gian các lần chạy trước (mỗi epoch khoảng 82 đến 87 giây cho EfficientNet/ResNet18/MobileNetV3 trên Colab T4). **Mỗi cell dưới đây là một fold**, nên chạy xong fold nào là có kết quả fold đó; nếu Colab ngắt giữa chừng, chạy lại từ fold dở. **Cần `git push` trước** (cell 3 chạy `git pull`).

## A. Chuẩn bị môi trường

### Cell 1: kiểm tra GPU
Phải thấy tên GPU (thường là `Tesla T4`).

In [ ]:
import torch
assert torch.cuda.is_available(), 'Bat GPU: Runtime -> Change runtime type -> T4 GPU'
print(torch.cuda.get_device_name(0))

### Cell 2: nối Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Cell 3: tải code từ GitHub và cài thư viện
Dòng cuối in commit đang dùng. Cần code có cờ `--cv_folds` và `--cv_fold` (đã có từ các lần trước).

In [ ]:
%cd /content
!git clone https://github.com/ColorOfDreams/dds-edge.git 2>/dev/null || echo 'already cloned'
%cd /content/dds-edge
!git pull -q
!pip install -q timm onnx onnxruntime tabulate
!pip install -q -e . --no-deps
!git log --oneline -1

### Cell 4: giải nén dữ liệu và nối vào đúng chỗ
**Kết quả cần thấy:** `20007 images; {'test': 4, 'train': 18, 'val': 4} drivers per split`.

In [ ]:
# data: extract to the local disk (much faster than reading from Drive), then link to data/processed/sf_320x240
# accepts sf_320x240*.zip (preferred) or sf_320x240*.rar in My Drive
import os, glob, shutil, zipfile, subprocess
os.chdir('/content/dds-edge')
if not glob.glob('/content/sf_data/**/manifest.csv', recursive=True):
    arch = None
    for ext in ('zip', 'rar'):
        found = (glob.glob(f'/content/drive/MyDrive/sf_320x240*.{ext}')
                 or glob.glob(f'/content/drive/MyDrive/**/sf_320x240*.{ext}', recursive=True))
        if found:
            arch = found[0]
            break
    assert arch, 'Khong thay sf_320x240*.zip/.rar trong My Drive: da upload len Drive chua?'
    print('archive:', arch)
    shutil.rmtree('/content/sf_data', ignore_errors=True)
    os.makedirs('/content/sf_data')
    if arch.endswith('.zip'):
        with zipfile.ZipFile(arch) as z:
            for info in z.infolist():
                name = info.filename.replace(chr(92), '/')  # zips made on Windows use backslashes
                if name.endswith('/'):
                    continue
                dst = os.path.join('/content/sf_data', name)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                with z.open(info) as f, open(dst, 'wb') as g:
                    shutil.copyfileobj(f, g)
    else:  # .rar needs the unrar tool
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'unrar'], check=True)
        subprocess.run(['unrar', 'x', '-idq', '-y', arch, '/content/sf_data/'], check=True)
hits = glob.glob('/content/sf_data/**/manifest.csv', recursive=True)
assert hits, 'Khong thay manifest.csv sau khi giai nen: ' + str(os.listdir('/content/sf_data')[:5])
src = os.path.dirname(hits[0])
os.makedirs('data/processed', exist_ok=True)
link = 'data/processed/sf_320x240'
if os.path.islink(link):  # stale link from an earlier run
    os.unlink(link)
assert not os.path.exists(link), f'{link} is a real folder, remove it first'
os.symlink(src, link)
import pandas as pd
m = pd.read_csv(link + '/manifest.csv')
print(len(m), 'images;', m.groupby('split')['group'].nunique().to_dict(), 'drivers per split')  # expect 20007; test 4, train 18, val 4

### Cell 5: lưu kết quả train vào Drive
`runs/` nối vào `MyDrive/dds-edge-runs/`. Mỗi fold ghi một thư mục run riêng (ví dụ `mnv3_f10_cv0`).

In [ ]:
# write runs/ straight to Drive so results survive a disconnect
%cd /content/dds-edge
!mkdir -p /content/drive/MyDrive/dds-edge-runs
!rm -rf runs
!ln -s /content/drive/MyDrive/dds-edge-runs runs
!ls -la runs | head -3

## B. Xem cách chia fold

### Cell 6: danh sách tài xế test của 5 fold
In, cho mỗi fold: tài xế nào làm test/val, số ảnh mỗi tập, và kiểm tra **mỗi tài xế làm test đúng một lần**. Cách chia này **không phụ thuộc mô hình**, nên ba mô hình đều dùng đúng các fold này.

In [ ]:
import pandas as pd
import sys
sys.path.insert(0, '/content/dds-edge/src')
from dds_edge.dl.common import cv_group_split
m = pd.read_csv('/content/dds-edge/data/processed/sf_320x240/manifest.csv')
tested = []
for f in range(5):
    gs = cv_group_split(m['group'], 5, f)
    d = m.assign(s=m['group'].map(gs))
    n = d.groupby('s').size().to_dict()
    t = sorted(g for g, v in gs.items() if v == 'test'); tested += t
    print(f"fold {f}: test {t}  | ảnh train {n['train']}, val {n['val']}, test {n['test']}")
print('\nmỗi tài xế làm test đúng một lần:', sorted(tested) == sorted(m['group'].unique()))

## C. MobileNetV3-Small, 5 fold

Mỗi cell chạy **một fold** (khoảng 15 phút). Dòng đầu mỗi lần chạy in danh sách tài xế `train/val/test` của fold đó; dòng JSON cuối có `n_params` (kiểm tra đúng mô hình) và `test_macro_f1` của fold. **Cần cả hai cờ** `--cv_folds 5` (tổng số fold) và `--cv_fold i` (fold nào làm test); gõ nhầm tên cờ (ví dụ `--fold`) sẽ báo `unrecognized arguments`.

### Cell 7: mnv3_f10_cv0 (fold 0)

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch mobilenet_v3_small --cv_folds 5 --cv_fold 0 --name mnv3_f10_cv0

### Cell 8: mnv3_f10_cv1 (fold 1)

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch mobilenet_v3_small --cv_folds 5 --cv_fold 1 --name mnv3_f10_cv1

### Cell 9: mnv3_f10_cv2 (fold 2)

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch mobilenet_v3_small --cv_folds 5 --cv_fold 2 --name mnv3_f10_cv2

### Cell 10: mnv3_f10_cv3 (fold 3)

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch mobilenet_v3_small --cv_folds 5 --cv_fold 3 --name mnv3_f10_cv3

### Cell 11: mnv3_f10_cv4 (fold 4)

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch mobilenet_v3_small --cv_folds 5 --cv_fold 4 --name mnv3_f10_cv4

## D. ResNet18, 5 fold

Mỗi cell chạy **một fold** (khoảng 8 phút). Dòng đầu mỗi lần chạy in danh sách tài xế `train/val/test` của fold đó; dòng JSON cuối có `n_params` (kiểm tra đúng mô hình) và `test_macro_f1` của fold. **Cần cả hai cờ** `--cv_folds 5` (tổng số fold) và `--cv_fold i` (fold nào làm test); gõ nhầm tên cờ (ví dụ `--fold`) sẽ báo `unrecognized arguments`.

### Cell 12: resnet18_f10_cv0 (fold 0)

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch resnet18 --cv_folds 5 --cv_fold 0 --name resnet18_f10_cv0

### Cell 13: resnet18_f10_cv1 (fold 1)

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch resnet18 --cv_folds 5 --cv_fold 1 --name resnet18_f10_cv1

### Cell 14: resnet18_f10_cv2 (fold 2)

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch resnet18 --cv_folds 5 --cv_fold 2 --name resnet18_f10_cv2

### Cell 15: resnet18_f10_cv3 (fold 3)

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch resnet18 --cv_folds 5 --cv_fold 3 --name resnet18_f10_cv3

### Cell 16: resnet18_f10_cv4 (fold 4)

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch resnet18 --cv_folds 5 --cv_fold 4 --name resnet18_f10_cv4

## E. (Tuỳ chọn) ResNet50, 5 fold
### Cell 17
Rất lâu (ResNet50 nặng nhất); **chỉ làm nếu còn nhiều giờ**. Bỏ `#` ở các dòng muốn chạy.

In [ ]:
%cd /content/dds-edge
# !python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch resnet50 --cv_folds 5 --cv_fold 0 --name resnet50_f10_cv0
# !python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch resnet50 --cv_folds 5 --cv_fold 1 --name resnet50_f10_cv1
# !python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch resnet50 --cv_folds 5 --cv_fold 2 --name resnet50_f10_cv2
# !python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch resnet50 --cv_folds 5 --cv_fold 3 --name resnet50_f10_cv3
# !python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch resnet50 --cv_folds 5 --cv_fold 4 --name resnet50_f10_cv4

## F. Tổng hợp

### Cell 18: trung bình ± độ lệch chuẩn của từng mô hình
Với mỗi mô hình (tiền tố tên run), đọc 5 fold và in **trung bình, độ lệch chuẩn, nhỏ nhất, lớn nhất** của test macro-F1 và accuracy, cùng điểm từng fold. Run nào chưa đủ 5 fold sẽ báo số fold hiện có.

**Cách đọc:** hai mô hình có khoảng "trung bình ± độ lệch chuẩn" chồng lên nhau nhiều thì chưa phân biệt được; xem cell 19 để so sánh theo cặp chặt hơn.

In [ ]:
import json, os
import numpy as np
RUNS = '/content/dds-edge/runs'
models = {'EfficientNet-Lite0': 'effl0_f10_cv', 'MobileNetV3-Small': 'mnv3_f10_cv', 'ResNet18': 'resnet18_f10_cv', 'ResNet50': 'resnet50_f10_cv'}
scores = {}
for name, pre in models.items():
    f1 = {}
    for i in range(5):
        p = f'{RUNS}/{pre}{i}/metrics.json'
        if os.path.exists(p):
            f1[i] = json.load(open(p))['test_macro_f1']
    if f1:
        scores[name] = f1
        v = np.array(list(f1.values()))
        sd = v.std(ddof=1) if len(v) > 1 else float('nan')
        print(f"{name:20s} {len(v)} fold  macro-F1 = {v.mean():.3f} ± {sd:.3f}  (min {v.min():.3f}, max {v.max():.3f})   từng fold: " + ' '.join(f'{x:.3f}' for x in v))
    else:
        print(f'{name:20s} chưa có fold nào')

### Cell 19: so sánh theo cặp trên cùng các fold
Với mỗi cặp mô hình, tính **hiệu số macro-F1 trên từng fold** (cùng tài xế test), rồi trung bình và độ lệch chuẩn của hiệu số, và **số fold mà mô hình thứ nhất thắng**. Quy tắc đọc: nếu trung bình hiệu số lớn hơn khoảng **2 lần độ lệch chuẩn / √5** (sai số chuẩn) và mô hình thắng ở 5/5 fold thì khác biệt có khả năng thật; nếu hiệu số đổi dấu giữa các fold thì chỉ là nhiễu. Chỉ so được các cặp có đủ 5 fold.

In [ ]:
import itertools
full = {k: v for k, v in scores.items() if len(v) == 5}
print(f"{'cặp':44s}{'hiệu số TB':>11s}{'sai số chuẩn':>13s}{'thắng':>7s}  kết luận")
for a, b in itertools.combinations(full, 2):
    d = np.array([full[a][i] - full[b][i] for i in range(5)])
    se = d.std(ddof=1) / np.sqrt(5)
    win = int((d > 0).sum())
    real = abs(d.mean()) > 2 * se and (win in (0, 5))
    print(f"{a + ' - ' + b:44s}{d.mean():+11.3f}{se:13.3f}{win:>5d}/5  {'có vẻ thật' if real else 'chưa phân biệt được (nhiễu)'}")
if len(full) < 2:
    print('cần ít nhất 2 mô hình có đủ 5 fold')

## Sau khi xong
1. Kết quả đã ở Drive `dds-edge-runs/`. Tải các thư mục `mnv3_f10_cv0..4` và `resnet18_f10_cv0..4` về máy (đợi Drive đồng bộ xong).
2. Trên máy bạn: `python -m dds_edge.dl.cv_summary --prefix mnv3_f10_cv --runs_dir <thư mục>` (tương tự cho `resnet18_f10_cv`) cho bảng chi tiết có F1 từng lớp.
3. Báo kết quả để cập nhật slide và bài báo: dùng **trung bình ± độ lệch chuẩn qua 5 fold** làm điểm chính thay cho một lần chia.